# Running the model

## Postglacial rebound

To run the accompanying `postglacial_rebound.prm` file simulating the post-glacial rebound, described in the [previous notebook](./3_modeling_plate_bending.ipynb), execute the following line with ASPECT:

In [ ]:
%%capture
! aspect-resdfsglease prm_files/postglacial_rebound.prm


> In the above cell, it is assumed that the ASPECT directory is installed system-wide or that you are using the HubZero ASPECT installation. If not, modify the ASPECT executable to the location where it is installed. If you are unsure whether ASPECT is running or not, delete the %%capture line and try again.

> ASPECT can be configured in either "release" mode or "debug" mode. Release mode is more optimized than debug mode, and will usually run much faster but lacks debugging symbols and more descriptive error messages. If you are making changes to ASPECT or altering parameter files, it might be a good idea to try running in debug mode in order to resolve any problems. The above code is calling the release version using `aspect-release` . If you want to try running debug mode, call `aspect-debug` instead.

<div class="alert alert-block alert-warning">
    <b>Note:</b> The provided .prm files are configured to run with the development version of ASPECT. This is because the implemented solvers for the elastic rheologies are not available under a tagged release yet. We provide the corresponding .gif files for the results of the simulations, but if you want to run the simulations yourself, you would need to install the development version of ASPECT available here: https://github.com/geodynamics/aspect.
</div>

<div class="alert alert-block alert-info">
    <b>Note:</b> If you are using the HubZero environment, please uncomment the following line to make sure that the required python packages are installed.
</div>

In [ ]:
# !pip install cmcrameri meshio


In [ ]:
# Load the relevant libraries
import matplotlib.pyplot as plt
from IPython.display import HTML
import cmcrameri.cm as cmc
import glob

import os
os.environ['VTK_DEFAULT_OPENGL_WINDOW'] = 'vtkEGLRenderWindow' # Suppresses a warning with Pyvista
import pyvista as pv

pv.global_theme.allow_empty_mesh = True

import sys
from pathlib import Path
# add the utilities file present in the source directory path
# The following path represents the parent directory of all
# gem-notebooks here : https://github.com/geodynamics-hubzero/gem-notebooks/tree/main
source_dir = '../../tools/'

# Get absolute path to avoid issues with relative paths
abs_path = str(Path(source_dir).resolve())

if abs_path not in sys.path:
    sys.path.append(abs_path)

import utilities


<div class="alert alert-block alert-info">
    <b>Note:</b> If you are using the HubZero environment, please uncomment the following lines to spin up a virtual X server to run the pyvista plotting commands.
</div>

In [ ]:
# pv.start_xvfb()


# Plotting the solution field

## Stress Field
We first look how the modeled normal stresses in the vertical direction evolve over time (animation below). These stresses provide information into the regions of the model that undergo compression (positive) and extension (negative streses) as the applied ice load deforms the elastic lithosphere. 

In [ ]:
def plot_deformation(output_dir, field, gif_file, cmap, clim):
    '''
    Process a series of VTU files in the given output directory, returning the 'field'
    data for plotting.
    Inputs:
    -----------
    output_dir : str
        Path to the folder containing solution VTU files (expected in solution/solution-*.vtu)
    field : str
        Name of the field we want to look at in the solution
    label : str
        Label for the field being plotted
    cmap  : str
        Colorscale to use for plotting the field
    clim : list
        Color limits for the field being plotted
    '''

    solution_file_names = sorted(glob.glob(output_dir + 'solution/solution-*.pvtu'))

    mesh = pv.read(solution_file_names[0])

    plotter = pv.Plotter()
    plotter.window_size = [1000, 400]  
    plotter.set_scale(0.25, 0.5, 1.0)

    # Create the first frame
    plotter.add_mesh(mesh, scalars=field, show_scalar_bar=False)
    plotter.hide_axes()

    # We know that the model is XY plane so the following
    # lines just sets up the camera
    plotter.view_xy()
    plotter.camera.roll = 0
    plotter.camera.parallel_projection = True
    plotter.camera.zoom(2)

    sargs_1 = dict(width=0.5, vertical=False, position_x=0.25, position_y=0.05, n_labels=5)

    plotter.open_gif(gif_file, fps=4)

    # Update function for each frame
    def update(frame):
        frame_int = int(frame)
        new_mesh = pv.read(solution_file_names[frame_int])
        plotter.clear()

        plotter.add_mesh(new_mesh, scalars=field, cmap=cmap, scalar_bar_args=sargs_1, clim=clim)
        time = frame_int * 1000 
        
        plotter.add_text(f"Time: {time:.2f} years", position="upper_left", font_size=12, color="black")

    n_frames = len(solution_file_names)

    for i in range(n_frames):
        update(i)
        plotter.write_frame()

    plotter.close()


In [ ]:
gif_name  = 'images/postglacial_rebound.gif'
plot_deformation('postglacial-rebound/', 've_stress_yy', gif_name, 'cmc.vik', [-2e6, 2e6])


<img src="images/postglacial_rebound.gif" alt="gif of postglacial rebound" width="600" align="center">

As the ice load is applied, we note compressive stresses in the center of the model domain (i.e., where the load is applied) and extensional stresses along the sides due to the upward bulge as the lithosphere bends in the center. This stress distribution is reversed when the load is removed and the lithosphere rebounds. Eventually, the model reaches equilibrium and the stresses do not change significantly.

## Topography field

In [ ]:
def plot_statistics_file (output_dir, field):
    '''
    This function plots the time evolution in the output statistics file created by ASPECT.
    
    Parameters:
    ----------
      output_dir : str
         Directory path where the statistics file is located
      field : str
         Name of the field to plot (e.g., 'RMS Velocity' or 'Max Velocity')
    '''

    statistics = output_dir + "statistics"
    data_stats = utilities.read_statistics(statistics)
    
    # if we want to look at the different columns uncomment the following line
    # data_stats.head()
    
    plt.ion()
    # Since we are interested in the convective vigor, let's plot the
    # RMS velocity and max velocity over time in our model
    fig, ax2 = plt.subplots(figsize=(5, 4))

    ax2.plot(data_stats["Time (years)"]/1e3, data_stats[field])
    ax2.set_xlabel("Time (kyr)")
    ax2.set_ylabel(field)


In [ ]:
plot_statistics_file('postglacial-rebound/', 'Maximum topography (m)')


<img src='./images/maximum_topography_postglacial.png' align="center"/>

plot_statistics_file('postglacial-rebound/', 'Maximum topography (m)')


## Ocean Island Loading

To run the accompanying `ocean_island_loading.prm` file simulating the elastic deformation due to the weight of ocean islands, described in the [previous notebook](./3_modeling_plate_bending.ipynb), execute the following line with ASPECT:

In [ ]:
%%capture
! aspect-release prm_files/ocean_island_loading.prm


In [ ]:
output_dir_1 = 'ocean-island-loading/'
plot_statistics_file(output_dir_1, 'Maximum topography (m)')


<img src='./images/maximum_topography_island.png' align="center"/>

Similar to the case above, the elastic lithosphere bends under the applied load and develops a side bulge that increases with time until the underlying mantle flow reaches equilibrium, approximately around 40000 years.

### Exercises

You can play with the following `Material Model` parameters to see how the onset of developed topography changes over time :

- Elastic moduli of the lithosphere (last term in `set Elastic shear moduli`) 
- Modify the viscosity of the lithospheric mantle (first term in `set Viscosities`)

<div style="text-align: right"> 
<img src="../../../assets/education-gem-notebooks_icon.png" alt="icon"  width="4%">
</div>